In [0]:
# Imports

from pyspark.sql.functions import current_timestamp, current_date
import requests

In [0]:
# Preparação dos endereços das tabelas

catalog = "atividade_2"

bronze = "bronze"
silver = "silver"
gold = "gold"

bronze_schema = f"{catalog}.{bronze}"
silver_schema = f"{catalog}.{silver}"
gold_schema = f"{catalog}.{gold}"

landing_path = f"/Volumes/{catalog}/{bronze}/Landing"

print(catalog)
print(bronze_schema)
print(silver_schema)
print(gold_schema)
print(landing_path)

In [0]:
# Criação do catálogo, dos esquemas e do volume de landing

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.Landing")

print("Catálogo, esquemas e volume de landing criados")

In [0]:
# Verificação do carregamento dos arquivos

movies_info_file = "movies_info_TMDB_IMDB.csv" # correção do nome, inconsistente no pdf
movies_financials_file = "movies_financials_IMDB_TMDB.csv"
movies_metrics_file = "movies_metrics_IMDB_TMDB.csv" # correção do nome, inconsistente no pdf
credits_and_tags_file = "credits_and_tags_IMDB_TMDB.csv"
movies_reviews_file = "movies_reviews.csv"

expected_files = [
    movies_info_file,
    movies_financials_file,
    movies_metrics_file,
    credits_and_tags_file,
    movies_reviews_file
]

try:
    existing = {f.name for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    existing = {}
    print(f"[ERRO]: {e}")

missing = [f for f in expected_files if f not in existing]

if missing:
    print("[ERRO] Alguns arquivos ainda não foram carregados:")
    for file in missing:
        print(f" - {file}")
else:
    print("[SUCESSO] Todos os arquivos foram carregados!")
    for file in dbutils.fs.ls(landing_path):
        print(f" - {file.name}: {file.size / 1024:.1f}KB")

In [0]:
# Lendo os arquivos com o Spark

df_movies_info = spark.read.csv(f"{landing_path}/{movies_info_file}", header=True, inferSchema=True)
df_movies_financials = spark.read.csv(f"{landing_path}/{movies_financials_file}", header=True, inferSchema=True)
df_movies_metrics = spark.read.csv(f"{landing_path}/{movies_metrics_file}", header=True, inferSchema=True)
df_credits_and_tags = spark.read.csv(f"{landing_path}/{credits_and_tags_file}", header=True, inferSchema=True)
df_movies_reviews = spark.read.csv(f"{landing_path}/{movies_reviews_file}", header=True, inferSchema=True)

In [0]:
# Gravando os dados na camada bronze

df_movies_info.withColumn("ingestion_datetime", current_timestamp()) \
              .write.format("delta").mode("append") \
              .saveAsTable(f"{bronze_schema}.tb_movies_info")

df_movies_financials.withColumn("ingestion_datetime", current_timestamp()) \
                    .write.format("delta").mode("append") \
                    .saveAsTable(f"{bronze_schema}.tb_movies_financials")

df_movies_metrics.withColumn("ingestion_datetime", current_timestamp()) \
                 .write.format("delta").mode("append") \
                 .saveAsTable(f"{bronze_schema}.tb_movies_metrics")

df_credits_and_tags.withColumn("ingestion_datetime", current_timestamp()) \
                   .write.format("delta").mode("append") \
                   .saveAsTable(f"{bronze_schema}.tb_credits_and_tags")

df_movies_reviews.withColumn("ingestion_datetime", current_timestamp()) \
                 .write.format("delta").mode("append") \
                 .saveAsTable(f"{bronze_schema}.tb_movies_reviews")

print("Dados carregados na camada bronze com sucesso!")

In [0]:
start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")

print(f"Carregando valor do dólar de {start_date} a {end_date}")

api_url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{start_date}'&@dataFinalCotacao='{end_date}'"
    "&$format=json" # select não consta na documentação https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/documentacao
)

response = requests.get(api_url)

if response.status_code == 200:
    dollar = response.json()["value"]

    df_dollar = spark.createDataFrame(dollar)
    print(f"Cotação do dólar carregada com sucesso: {df_dollar.count()} linhas")
else:
    df_dollar = spark.createDataFrame([
        (5.1803, 5.1809, "2026-09-30 13:11:44.783262"),
        (5.2073, 5.2079, "2026-10-01 13:10:35.4469"),
        (5.2232, 5.2238, "2026-10-02 13:03:16.256632"),
        (4.9853, 4.9859, "2026-10-05 13:07:36.558602"),
        (4.9692, 4.9698, "2026-10-06 13:03:21.267287")
    ], schema='cotacaoCompra double, cotacaoVenda double, dataHoraCotacao string')
    
    print(f"Cotação do dólar carregada com dados de teste!")

In [0]:
df_dollar.withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_cotacao_dolar")

print("Cotação do dólar salva com sucesso")